# GRACE-FO 单程光行时延计算（简化版）

这版只保留当前任务需要的内容：

- 仅计算 **D → C** 单程；
- 仅保留 **TPM** 和 **TSM**；
- **完全去掉 THM**；
- 所有参数都在开头 `CONFIG` 里直接设置；
- 不使用 `argparse`；
- 不读取 `.gfc` 高阶球谐系数，只直接使用地球 **GM** 和 **Re**；
- 输出为 **xlsx**。

如果你本地文件名不同，只改第一个 code cell 里的 `CONFIG` 即可。


In [1]:

# ==============================
# 1) 运行参数：只在这里改
# ==============================

CONFIG = {
    # 输入文件
    "c_file": r"GNI1B_2022-06-05_C_04.txt",
    "d_file": r"GNI1B_2022-06-05_D_04.txt",

    # 地球常数（直接写在这里，不再读取 gfc 文件）
    # 来自 EIGEN-6C4 头信息：
    # earth_gravity_constant = 0.3986004415E+15
    # radius                 = 0.6378136460E+07
    "GM": 0.3986004415E+15,      # m^3 / s^2
    "Re": 0.6378136460E+07,      # m

    # 主迭代控制
    "tol": 1e-18,
    "max_iter": 3,

    # 只取前若干条记录；None 表示全算
    "max_rows": None,

    # 是否显示进度条
    "show_progress": True,

    # 输出文件
    "out_xlsx": "ltc_dc_tpm_tsm_only.xlsx",
    "out_history_json": "ltc_dc_tpm_tsm_history_first5.json",
}


In [2]:

# ==============================
# 2) 导入与基础函数
# ==============================

from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


C0 = 299792458.0
OMEGA_E_DEFAULT = np.array([0.0, 0.0, 7.2921150e-5], dtype=float)


def progress_iter(iterable, total: int, desc: str, enable: bool = True):
    """Notebook 里的简单进度显示。"""
    if enable and tqdm is not None:
        return tqdm(iterable, total=total, desc=desc)
    return iterable


def norm3(x: np.ndarray) -> float:
    return float(np.linalg.norm(np.asarray(x, dtype=float)))


def unit3(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    n = norm3(x)
    if n == 0.0:
        raise ValueError("零向量无法单位化。")
    return x / n


def second_order_taylor_position(r_tr: np.ndarray,
                                 v_tr: np.ndarray,
                                 a_tr: np.ndarray,
                                 delta_t: float) -> np.ndarray:
    """
    r(tr - dt) ≈ r(tr) - v(tr) * dt + 0.5 * a(tr) * dt^2
    """
    r_tr = np.asarray(r_tr, dtype=float)
    v_tr = np.asarray(v_tr, dtype=float)
    a_tr = np.asarray(a_tr, dtype=float)
    dt = float(delta_t)
    return r_tr - v_tr * dt + 0.5 * a_tr * dt**2


@dataclass
class ModelConstants:
    GM: float
    Re: float


@dataclass
class OneWayIterationResult:
    delta_t: float
    history: List[Dict[str, float]]


In [3]:
# ==============================
# 3) 数据读取与预处理
# ==============================

def read_gni1b_txt(path: str | Path) -> pd.DataFrame:
    """
    读取 GNI1B 文本文件。
    仅保留当前任务需要的列：
      gps_time, coord_ref, xpos, ypos, zpos, xvel, yvel, zvel
    """
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"未找到文件: {path}")

    lines = path.read_text(encoding="utf-8", errors="ignore").splitlines()
    try:
        end_idx = next(i for i, line in enumerate(lines) if line.strip() == "# End of YAML header")
    except StopIteration as exc:
        raise ValueError(f"文件 {path} 未找到 '# End of YAML header'。") from exc

    col_names = [
        "gps_time", "sat_id", "coord_ref",
        "xpos", "ypos", "zpos",
        "xpos_err", "ypos_err", "zpos_err",
        "xvel", "yvel", "zvel",
        "xvel_err", "yvel_err", "zvel_err",
        "qualflg",
    ]

    df = pd.read_csv(
        path,
        delim_whitespace=True,
        skiprows=end_idx + 1,
        header=None,
        names=col_names,
    )

    df = df[["gps_time", "coord_ref", "xpos", "ypos", "zpos", "xvel", "yvel", "zvel"]].copy()

    # 当前任务默认使用惯性系 GNI1B（coord_ref == 'I'）
    bad = df.loc[df["coord_ref"] != "I"]
    if not bad.empty:
        raise ValueError(f"文件 {path} 中存在非惯性系记录，coord_ref 应为 'I'。")

    return df.reset_index(drop=True)


def finite_difference_weights(x: np.ndarray, x0: float, deriv_order: int) -> np.ndarray:
    """
    Fornberg 型有限差分权重，支持非均匀节点。
    返回在节点 x 上，对 x0 处 deriv_order 阶导数的权重。
    """
    x = np.asarray(x, dtype=float)
    n = x.size
    m = int(deriv_order)

    if n == 0:
        raise ValueError("x 不能为空。")
    if m < 0:
        raise ValueError("deriv_order 必须 >= 0。")
    if m >= n:
        raise ValueError("节点数必须大于导数阶数。")

    c = np.zeros((n, m + 1), dtype=float)
    c[0, 0] = 1.0
    c1 = 1.0
    c4 = x[0] - x0

    for i in range(1, n):
        mn = min(i, m)
        c2 = 1.0
        c5 = c4
        c4 = x[i] - x0

        for j in range(i):
            c3 = x[i] - x[j]
            if c3 == 0.0:
                raise ValueError("有限差分节点重复，无法计算权重。")
            c2 *= c3

            if j == i - 1:
                for k in range(mn, 0, -1):
                    c[i, k] = c1 * (k * c[i - 1, k - 1] - c5 * c[i - 1, k]) / c2
                c[i, 0] = -c1 * c5 * c[i - 1, 0] / c2

            for k in range(mn, 0, -1):
                c[j, k] = (c4 * c[j, k] - k * c[j, k - 1]) / c3
            c[j, 0] = c4 * c[j, 0] / c3

        c1 = c2

    return c[:, m]


def high_order_first_derivative(t: np.ndarray,
                                y: np.ndarray,
                                stencil: int = 5) -> np.ndarray:
    """
    用高阶有限差分计算一阶导数 dy/dt。

    处理策略：
    - 优先使用 5 点模板；
    - 靠近边界时自动使用单边/偏置模板；
    - 若点数不足，则自动退化为可用的最高阶模板；
    - 支持非均匀时间节点。
    """
    t = np.asarray(t, dtype=float)
    y = np.asarray(y, dtype=float)

    if t.ndim != 1 or y.ndim != 1:
        raise ValueError("t 和 y 必须为 1 维数组。")
    if t.size != y.size:
        raise ValueError("t 和 y 长度必须一致。")
    if t.size < 3:
        raise ValueError("至少需要 3 个点才能计算导数。")

    n = t.size
    stencil = int(max(3, stencil))
    stencil = min(stencil, n)
    if stencil % 2 == 0:
        stencil -= 1
    stencil = max(3, stencil)

    half = stencil // 2
    dydt = np.empty(n, dtype=float)

    for i in range(n):
        left = max(0, i - half)
        right = min(n, i + half + 1)

        if right - left < stencil:
            if left == 0:
                right = min(n, left + stencil)
            elif right == n:
                left = max(0, n - stencil)

        x_stencil = t[left:right]
        y_stencil = y[left:right]
        w = finite_difference_weights(x_stencil, t[i], 1)
        dydt[i] = float(np.dot(w, y_stencil))

    return dydt


def estimate_acceleration_from_velocity(df: pd.DataFrame) -> pd.DataFrame:
    """
    GNI1B 文本里没有直接提供加速度。
    这里按与前面程序相同的方式，使用高阶有限差分：
        a = dv / dt
    """
    out = df.copy()

    t = out["gps_time"].to_numpy(dtype=float)
    vx = out["xvel"].to_numpy(dtype=float)
    vy = out["yvel"].to_numpy(dtype=float)
    vz = out["zvel"].to_numpy(dtype=float)

    out["xacc"] = high_order_first_derivative(t, vx, stencil=5)
    out["yacc"] = high_order_first_derivative(t, vy, stencil=5)
    out["zacc"] = high_order_first_derivative(t, vz, stencil=5)
    return out


def prepare_satellite_dataframe(path: str | Path) -> pd.DataFrame:
    df = read_gni1b_txt(path)
    df = estimate_acceleration_from_velocity(df)
    return df

In [4]:

# ==============================
# 4) TPM / TSM 计算公式
# ==============================

def compute_tpm(rr_gcrs: np.ndarray,
                re_gcrs: np.ndarray,
                GM: float,
                c0: float = C0) -> float:
    """
    T_PM = 2GM/c^3 * ln((|r_r| + |r_e| + |r_r-r_e|) / (|r_r| + |r_e| - |r_r-r_e|))
    """
    rrn = norm3(rr_gcrs)
    ren = norm3(re_gcrs)
    dr = norm3(rr_gcrs - re_gcrs)

    num = rrn + ren + dr
    den = rrn + ren - dr
    if den <= 0.0:
        raise ValueError("T_PM 计算失败：对数分母 <= 0。")

    return 2.0 * GM / c0**3 * np.log(num / den)


def compute_tsm(rr_gcrs: np.ndarray,
                re_gcrs: np.ndarray,
                d0_inst: np.ndarray,
                delta_t_sr: float,
                omega_e_vec: np.ndarray,
                GM: float,
                Re: float,
                c0: float = C0) -> float:
    """
    T_SM ≈ (2GM Re^2 / 5c^3) * ((ω × r_e) · d0) * (1/|r_e|^3 + 1/|r_r|^3) * Δt_SR
    """
    re_n = norm3(re_gcrs)
    rr_n = norm3(rr_gcrs)
    cross_term = np.cross(omega_e_vec, re_gcrs)
    dot_term = float(np.dot(cross_term, d0_inst))
    geom_term = (1.0 / re_n**3) + (1.0 / rr_n**3)

    return -(2.0 * GM * Re**2 / (5.0 * c0**3)) * dot_term * geom_term * delta_t_sr


In [5]:

# ==============================
# 5) 单个历元的严格迭代
# ==============================

def iterate_one_way_total_delay(
    tr_seconds: float,
    rA_tr: np.ndarray,
    vA_tr: np.ndarray,
    aA_tr: np.ndarray,
    rB_tr: np.ndarray,
    constants: ModelConstants,
    omega_e_vec: np.ndarray = OMEGA_E_DEFAULT,
    tol: float = 1e-18,
    max_iter:int = 20 ,
) -> OneWayIterationResult:
    """
    这里按你当前要求，只保留 TPM 和 TSM：

        Δt^(0) = Δt_inst

        先用 Δt^(n) 计算
            r_e^(n) = r_A(tr - Δt^(n))

        再计算
            T_PM^(n), T_SM^(n)

        然后更新
            Δt^(n+1) = |r_B(tr)-r_e^(n)|/c0 + T_PM^(n) + T_SM^(n)

    其中：
    - d0 取瞬时 LOS 方向
    - te 使用一阶近似
    - Δt_SR = tr - te
    - 用于 T_SM 的 Δt_SR 固定，不在主迭代里更新
    """
    rA_tr = np.asarray(rA_tr, dtype=float)
    vA_tr = np.asarray(vA_tr, dtype=float)
    aA_tr = np.asarray(aA_tr, dtype=float)
    rB_tr = np.asarray(rB_tr, dtype=float)

    rel_inst = rB_tr - rA_tr
    d0_inst = unit3(rel_inst)
    delta_t_inst = norm3(rel_inst) / C0

    te_first_order = tr_seconds - delta_t_inst - delta_t_inst * float(np.dot(d0_inst, vA_tr)) / C0
    delta_t_sr_fixed = tr_seconds - te_first_order

    delta_t_n = delta_t_inst
    history: List[Dict[str, float]] = []

    for n in range(max_iter):
        re_n = second_order_taylor_position(rA_tr, vA_tr, aA_tr, delta_t_n)
        rr_n = rB_tr

        tpm_n = compute_tpm(rr_n, re_n, constants.GM, C0)
        tsm_n = compute_tsm(
            rr_gcrs=rr_n,
            re_gcrs=re_n,
            d0_inst=d0_inst,
            delta_t_sr=delta_t_sr_fixed,
            omega_e_vec=omega_e_vec,
            GM=constants.GM,
            Re=constants.Re,
            c0=C0,
        )

        geom_n = norm3(rr_n - re_n) / C0
        delta_t_np1 = geom_n +tpm_n+ tsm_n

        history.append({
            "iter": int(n),
            "delta_t_n": float(delta_t_n),
            "delta_t_np1": float(delta_t_np1),
            "abs_update": float(abs(delta_t_np1 - delta_t_n)),
            "delta_t_inst": float(delta_t_inst),
            "te_first_order": float(te_first_order),
            "delta_t_sr_fixed": float(delta_t_sr_fixed),
            "geom_n": float(geom_n),
            "T_PM_n": float(tpm_n),
            "T_SM_n": float(tsm_n),
        })

        if abs(delta_t_np1 - delta_t_n) < tol:
            return OneWayIterationResult(delta_t=float(delta_t_np1), history=history)

        delta_t_n = delta_t_np1

    return OneWayIterationResult(delta_t=float(delta_t_n), history=history)


In [6]:
# ==============================
# 6) 整个时间序列：只算 C -> D
# ==============================

def compute_c_to_d_total_delay(
    c_file: str | Path,
    d_file: str | Path,
    constants: ModelConstants,
    tol: float = 1e-18,
    max_iter: int = 20,
    max_rows: Optional[int] = None,
    show_progress: bool = True,
) -> Tuple[pd.DataFrame, List[List[Dict[str, float]]]]:
    """
    仅计算 C -> D 单程时延。
    不计算 HM，不做多余参考系变换。
    """
    c_df = prepare_satellite_dataframe(c_file)
    d_df = prepare_satellite_dataframe(d_file)

    merged = pd.merge(
        c_df, d_df,
        on="gps_time",
        suffixes=("_tx", "_rx"),
        how="inner",
    ).sort_values("gps_time").reset_index(drop=True)

    if max_rows is not None:
        merged = merged.iloc[:max_rows].copy()

    out_rows = []
    histories_all: List[List[Dict[str, float]]] = []

    iterator = progress_iter(
        merged.iterrows(),
        total=len(merged),
        desc="C -> D 计算进度",
        enable=show_progress,
    )

    for _, row in iterator:
        tr = float(row["gps_time"])

        # 发射星 A = C
        rA = np.array([row["xpos_tx"], row["ypos_tx"], row["zpos_tx"]], dtype=float)
        vA = np.array([row["xvel_tx"], row["yvel_tx"], row["zvel_tx"]], dtype=float)
        aA = np.array([row["xacc_tx"], row["yacc_tx"], row["zacc_tx"]], dtype=float)

        # 接收星 B = D
        rB = np.array([row["xpos_rx"], row["ypos_rx"], row["zpos_rx"]], dtype=float)

        result = iterate_one_way_total_delay(
            tr_seconds=tr,
            rA_tr=rA,
            vA_tr=vA,
            aA_tr=aA,
            rB_tr=rB,
            constants=constants,
            omega_e_vec=OMEGA_E_DEFAULT,
            tol=tol,
            max_iter=max_iter,
        )

        last = result.history[-1]
        delta_t_inst = float(last["delta_t_inst"])

        out_rows.append({
            "gps_time": tr,
            "delta_t": result.delta_t,
            "delta_t_inst": delta_t_inst,
            "delta_t_minus_inst": float(result.delta_t - delta_t_inst),
            "geom": last["geom_n"],
            "T_PM": last["T_PM_n"],
            "T_SM": last["T_SM_n"],
            "n_iter": len(result.history),
            "last_abs_update": last["abs_update"],
        })
        histories_all.append(result.history)

    return pd.DataFrame(out_rows), histories_all

In [7]:

# ==============================
# 7) 写出结果
# ==============================

def write_results_to_xlsx(out_df: pd.DataFrame, out_xlsx: str | Path) -> None:
    out_xlsx = str(out_xlsx)
    with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
        out_df.to_excel(writer, sheet_name="results", index=False)


In [8]:
# ==============================
# 8) 主执行
# ==============================

constants = ModelConstants(
    GM=float(CONFIG["GM"]),
    Re=float(CONFIG["Re"]),
)

out_df, histories = compute_c_to_d_total_delay(
    c_file=CONFIG["c_file"],
    d_file=CONFIG["d_file"],
    constants=constants,
    tol=float(CONFIG["tol"]),
    max_iter=int(CONFIG["max_iter"]),
    max_rows=CONFIG["max_rows"],
    show_progress=bool(CONFIG["show_progress"]),
)

write_results_to_xlsx(out_df, CONFIG["out_xlsx"])

preview_hist = {
    "CD": histories[:5],
}
with open(CONFIG["out_history_json"], "w", encoding="utf-8") as f:
    json.dump(preview_hist, f, ensure_ascii=False, indent=2)

print("计算完成。")
print(f"结果文件: {CONFIG['out_xlsx']}")
print(f"历史文件: {CONFIG['out_history_json']}")
print("当前物理约定：发射星 = C，接收星 = D")
print("输出列：")
print(out_df.columns.tolist())

out_df.head()

C -> D 计算进度:   0%|          | 0/86400 [00:00<?, ?it/s]

计算完成。
结果文件: ltc_dc_tpm_tsm_only.xlsx
历史文件: ltc_dc_tpm_tsm_history_first5.json
当前物理约定：发射星 = C，接收星 = D
输出列：
['gps_time', 'delta_t', 'delta_t_inst', 'delta_t_minus_inst', 'geom', 'T_PM', 'T_SM', 'n_iter', 'last_abs_update']


,gps_time,delta_t,delta_t_inst,delta_t_minus_inst,geom,T_PM,T_SM,n_iter,last_abs_update
0,707659200.0,0.00065,0.00065,-1.655968e-08,0.00065,8.418995e-13,7.429549e-21,3,1.203464e-17
1,707659201.0,0.00065,0.00065,-1.655968e-08,0.00065,8.419007e-13,7.430574e-21,3,1.214306e-17
2,707659202.0,0.00065,0.00065,-1.655968e-08,0.00065,8.419020e-13,7.431599e-21,3,9.866240e-18
3,707659203.0,0.00065,0.00065,-1.655967e-08,0.00065,8.419032e-13,7.432625e-21,3,1.127570e-17
4,707659204.0,0.00065,0.00065,-1.655967e-08,0.00065,8.419044e-13,7.433650e-21,3,9.866240e-18
